In [1]:
def safe_file_io(filepath: str):
    content = "Python 蜕变计划 🚀\nHello, 世界！\n"
    
    # ✅ 1. 使用 with 语句（上下文管理器），确保无论是否异常都会关闭文件
    # ✅ 2. 显式指定 encoding='utf-8'，杜绝跨平台乱码
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)
        print("✅ 写入成功")
    
    # 读取
    with open(filepath, 'r', encoding='utf-8') as f:
        # 如果文件不大，直接 read()；如果大，用 for line in f:
        loaded_content = f.read()
        print(f"📖 读取内容:\n{loaded_content}")

safe_file_io("test_utf8.txt")

✅ 写入成功
📖 读取内容:
Python 蜕变计划 🚀
Hello, 世界！



In [2]:
from pathlib import Path
from datetime import datetime

# 获取当前脚本所在目录
current_dir = Path(__file__).parent.resolve()

# 面向对象的路径拼接：使用 / 运算符
logs_dir = current_dir / "logs"

# 创建目录（parents=True 类似 mkdir -p，exist_ok=True 避免已存在时报错）
logs_dir.mkdir(parents=True, exist_ok=True)

# 生成带时间戳的文件名
today = datetime.now().strftime("%Y%m%d")
log_file = logs_dir / f"app_{today}.log"

print(f"📁 目录已就绪: {logs_dir}")
print(f"📄 日志文件路径: {log_file}")
print(f"🔍 文件是否存在: {log_file.exists()}")
print(f"📏 文件后缀: {log_file.suffix}")  # .log
print(f"📛 文件主名: {log_file.stem}")    # app_20260708

NameError: name '__file__' is not defined

In [1]:
from pathlib import Path
def count_errors_in_large_file(filepath:Path) -> int:
    error_count = 0
    with open(filepath,'r',encoding='utf-8',errors='ignore') as f:
        for line in f:
            if 'ERROR' in line:
                error_count += 1
    return error_count


In [ ]:
import json
import csv
from pathlib import Path
from dataclasses import dataclass,asdict
@dataclass 
class User:
    name:str
    age:int
    role:str
users=[User("张三", 28, "Admin"), User("李四", 22, "User")]
json_path = Path('user.json')
with open(json_path,'w',encoding='utf-8') as f:
    json.dump([asdict(u) for u in users],f,ensure_ascii=False,indent=4)
with open(json_path,'r',encoding='utf-8') as f:
    loaded_users = json.load(f)
    print(f"JSON 读取: {loaded_users[0]['name']}")
csv_path = Path('users.csv')
with open(csv_path,'w',encoding='utf-8-sig',newline='') as f:
    writer = csv.DictWriter(f,fieldnames=['name','age','role'])
    writer.writeheader()
    writer.writerows(asdict(u) for u in users)
with open(csv_path,'r',encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"CSV 读取: {row['name']} - {row['role']}")


JSON 读取: 张三
CSV 读取: 张三 - Admin
CSV 读取: 李四 - User


In [2]:
import time
def safe_divide(a,b):
    start_time = time.perf_counter()
    try:
        result = a / b
    except ZeroDivisionError:
        print("❌ 错误：除数不能为 0")
        return None
    except TypeError as e:
        print(f"❌ 错误：类型不匹配 ({e})")
        return None
    else:
       print(f"✅ 计算成功: {result}")
       return result 
    finally:
        elapsed = time.perf_counter() - start_time
        print(f"⏱️ 耗时: {elapsed:.6f}s\n")
safe_divide(10,2)
safe_divide(10,0)
safe_divide(10,'a')


    


✅ 计算成功: 5.0
⏱️ 耗时: 0.000180s

❌ 错误：除数不能为 0
⏱️ 耗时: 0.000008s

❌ 错误：类型不匹配 (unsupported operand type(s) for /: 'int' and 'str')
⏱️ 耗时: 0.000006s



In [3]:
class PaymentError(Exception):
    pass
class InsufficientFundsError(PaymentError):
    def __init__(self,balance:float,amount:float):
        self.balance = balance
        self.amount = amount
        super().__init__(f"余额不足: 当前 {balance}，需要 {amount}")

class PaymentGateError(PaymentError):
    pass

def process_payment(user_balance:float,amount:float):
    if amount > user_balance:
        raise InsufficientFundsError(user_balance,amount)
    import random
    if random.random() < 0.2:
        raise PaymentGateError("支付宝网关超时")

try:
    process_payment(100.0,150.0)
except InsufficientFundsError as e:
    print(f"💸 业务提示: {e} (差额: {e.amount - e.balance})")
except PaymentGateError as e:
    print(f"🌐 系统提示: 支付通道繁忙，请稍后重试 ({e})")
except PaymentError as e:
    print(f"⚠️ 未知支付错误: {e}")




💸 业务提示: 余额不足: 当前 100.0，需要 150.0 (差额: 50.0)


In [12]:
import sqlite3
class DatabaseConnectionError(Exception):
    pass
def connect_to_db(db_path:str):
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM non_existent_table")
    except sqlite3.OperationalError as e:
        raise DatabaseConnectionError(f"数据库初始化失败: {db_path}") from e
try:
    connect_to_db(":memory:")
except DatabaseConnectionError as e:
    print(f"捕获到: {e}")
    print(f"原始原因 (__cause__): {e.__cause__}")


捕获到: 数据库初始化失败: :memory:
原始原因 (__cause__): no such table: non_existent_table


In [13]:
from pathlib import Path
from contextlib import suppress
def cleanup_temp_files(*file_paths:str):
    for path_str in file_paths:
        p = Path(path_str)
        with suppress(FileNotFoundError):
            p.unlink()
            print(f"🗑️ 已删除: {p}")
Path("temp1.txt").touch()
cleanup_temp_files("temp1.txt", "temp2_not_exist.txt")
    


🗑️ 已删除: temp1.txt


In [15]:
import tempfile
from pathlib import Path
def process_downloaded_image():
    with tempfile.NamedTemporaryFile(suffix='.jpg',delete=False) as tmp:
        tmp_path = Path(tmp.name)
        print(f"⬇️ 模拟下载到临时文件: {tmp_path}")
        tmp.write(b"FAKE_IMAGE_DATA_BINARY")
    try:
        print(f"⚙️ 正在处理 {tmp_path}，大小: {tmp_path.stat().st_size} bytes")
    finally:
        tmp_path.unlink(missing_ok=True)
        print(f"🧹 临时文件已清理")
process_downloaded_image()

⬇️ 模拟下载到临时文件: C:\Users\YUHAO~1.BIA\AppData\Local\Temp\tmpd6vklzeb.jpg
⚙️ 正在处理 C:\Users\YUHAO~1.BIA\AppData\Local\Temp\tmpd6vklzeb.jpg，大小: 22 bytes
🧹 临时文件已清理


In [17]:
import json
import time
from pathlib import Path
from typing import Any,Dict

class ConfigError(Exception): pass 
class ConfigNotFoundError(ConfigError): pass
class ConfigParseError(ConfigError): pass

class ConfigLoader:
    def __init__(self,config_path:str|Path):
        self.path = Path(config_path).resolve()
        self._cache:Dict[str,Any] = {}
        self._last_mtime:float = 0.0
    def load(self,force:bool = False) -> Dict[str,Any]:
        if not self.path.exists():
            raise ConfigNotFoundError(f"配置文件不存在: {self.path}")
        current_mtime = self.path.stat().st_mtime

        if not force and current_mtime ==self._last_mtime and self._cache:
            return self._cache
        try:
            with open(self.path,'r',encoding='utf-8') as f:
                self._cache = json.load(f)
                self._last_mtime = current_mtime
                print(f"✅ 配置已加载/更新: {self.path.name}")
                return self._cache
        except json.JSONDecodeError as e:
            raise ConfigParseError(f"配置文件格式损坏: {self.path}") from e
        
config_file = Path("app_config.json")
config_file.write_text('{"db_host": "localhost", "port": 5432}', encoding='utf-8')

loader = ConfigLoader(config_file)

try:
    cfg = loader.load()
    print(f"DB Host: {cfg['db_host']}")
    cfg2 = loader.load()
    
    time.sleep(0.1)
    config_file.write_text('{"db_host": "192.168.1.1", "port": 5432}', encoding='utf-8')
    cfg3 = loader.load()
    print(f"New DB Host: {cfg3['db_host']}")
except ConfigError as e:
   print(f"❌ 配置加载失败: {e}")
finally:
    config_file.unlink(missing_ok=True)

    

✅ 配置已加载/更新: app_config.json
DB Host: localhost
✅ 配置已加载/更新: app_config.json
New DB Host: 192.168.1.1
